In [ ]:
import os
from os.path import expanduser
home = expanduser("~/")

import sys
# sys.path.insert(0, '/global/u2/x/xshuang/gigalens-xh-dev/src')

# import sys
conda_env = sys.path[1]
del sys.path[1]

import os
# sys.path.append(f'{os.environ['HOME']}/gigalens_personal/gigalens/src')
sys.path.append(home+'/gigalens'+'/src')
sys.path.append(conda_env)
sys.path.append(home+'/GIGALens-Code/')
print(sys.path)



srcdir = os.path.join(home, "gigalens/src/")


In [ ]:
import tensorflow_probability.substrates.jax as tfp

from gigalens.jax.inference import ModellingSequence
from gigalens.jax.model import ForwardProbModel, BackwardProbModel
from gigalens.model import PhysicalModel
from gigalens.jax.simulator import LensSimulator
from gigalens.simulator import SimulatorConfig
from gigalens.jax.profiles.light import sersic
from gigalens.jax.profiles.mass import epl, shear

import jax
from jax import random
import numpy as np
import optax
from jax import numpy as jnp
from matplotlib import pyplot as plt
import optax
import corner
import yaml
import pickle
from helpers import *
import blackjax
import importlib
tfd = tfp.distributions

In [ ]:
# prior = make_default_prior()
# gigal_dir = os.path.join(home,'gigalens/src/gigalens/')
# kernel = np.load(gigal_dir + '/assets/psf.npy').astype(np.float32)
# sim_config = SimulatorConfig(delta_pix=0.065, num_pix=60, supersample=2, kernel=kernel)
# phys_model = PhysicalModel([epl.EPL(50), shear.Shear()], [sersic.SersicEllipse(use_lstsq=False)], [sersic.SersicEllipse(use_lstsq=False)])
# lens_sim = LensSimulator(phys_model, sim_config, bs=1)
# observed_img = np.load(gigal_dir + '/assets/demo.npy')
# prob_model = ForwardProbModel(prior, observed_img, background_rms=0.2, exp_time=100)
# model_seq = ModellingSequence(phys_model, prob_model, sim_config)

# results = {}
# results["MAP"] = MAPResults.load(os.path.join(home, "GIGALens-Code/alternate_inference/test_system"), model_seq)
# results["SVI"] = SVIResults.load(os.path.join(home, "GIGALens-Code/alternate_inference/test_system"), model_seq)
# results["HMC"] = HMCResults.load(os.path.join(home, "GIGALens-Code/alternate_inference/test_system"), model_seq)

In [ ]:
prior = make_default_prior()
kernel = np.load('/global/homes/l/linusu/gigalens/src/gigalens/assets/psf.npy').astype(np.float32)
sim_config = SimulatorConfig(delta_pix=0.065, num_pix=80, supersample=2, kernel=kernel)
phys_model = PhysicalModel([epl.EPL(50), shear.Shear()], [sersic.SersicEllipse(use_lstsq=False)], [sersic.SersicEllipse(use_lstsq=False)])
lens_sim = LensSimulator(phys_model, sim_config, bs=1)

systems_dir = os.path.join(home, "GIGALens-Code", "SystemSaves")
f = np.load(os.path.join(systems_dir, "100SystemsStandard80px.npz"))
keys = f.files
observed_imgs = jnp.array([f[key] for key in keys])
observed_img = observed_imgs[60]

prob_model = ForwardProbModel(prior, observed_img, background_rms=0.2, exp_time=100)
model_seq = ModellingSequence(phys_model, prob_model, sim_config)

results_dir = os.path.join(home, "sys60_converged_4e4_burnin")
results = {}
results["MAP"] = MAPResults.load(results_dir, model_seq)
results["SVI"] = SVIResults.load(results_dir, model_seq)
results["HMC"] = HMCResults.load(results_dir, model_seq)

In [ ]:
lens_prior = tfd.JointDistributionSequential(
    [
        tfd.JointDistributionNamed(
            dict(
                theta_E=tfd.LogNormal(jnp.log(1.25), 0.4),
                gamma=tfd.TruncatedNormal(2, 0.5, 1, 3),
                e1=tfd.Normal(0, 0.2),
                e2=tfd.Normal(0, 0.2),
                center_x=tfd.Normal(0, 0.06),
                center_y=tfd.Normal(0, 0.06),
            )
        ),
        tfd.JointDistributionNamed(
            dict(gamma1=tfd.Normal(0, 0.1), gamma2=tfd.Normal(0, 0.1))
        ),
    ]
)
lens_light_prior = tfd.JointDistributionSequential(
    [
        tfd.JointDistributionNamed(
            dict(
                R_sersic=tfd.LogNormal(jnp.log(1.6), 0.25),
                n_sersic=tfd.Uniform(0.5, 8),
                e1=tfd.TruncatedNormal(0, 0.1, -0.15, 0.15),
                e2=tfd.TruncatedNormal(0, 0.1, -0.15, 0.15),
                center_x=tfd.Normal(0, 0.02),
                center_y=tfd.Normal(0, 0.02),
                Ie=tfd.LogNormal(jnp.log(300.0), 0.5),
            )
        )
    ]
)

source_light_prior = tfd.JointDistributionSequential(
    [
        tfd.JointDistributionNamed(
            dict(
                R_sersic=tfd.LogNormal(jnp.log(0.25), 0.25),
                n_sersic=tfd.Uniform(0.5, 8),
                e1=tfd.TruncatedNormal(0, 0.3, -0.5, 0.5),
                e2=tfd.TruncatedNormal(0, 0.3, -0.5, 0.5),
                center_x=tfd.Normal(0, 0.5),
                center_y=tfd.Normal(0, 0.5),
                Ie=tfd.LogNormal(jnp.log(150.0), 0.9),
            )
        ),
        tfd.JointDistributionNamed(
            dict(
                R_sersic=tfd.LogNormal(jnp.log(0.25), 0.25),
                n_sersic=tfd.Uniform(0.5, 8),
                e1=tfd.TruncatedNormal(0, 0.3, -0.5, 0.5),
                e2=tfd.TruncatedNormal(0, 0.3, -0.5, 0.5),
                center_x=tfd.Normal(0, 0.5),
                center_y=tfd.Normal(0, 0.5),
                Ie=tfd.LogNormal(jnp.log(150.0), 0.9),
            )
        )

    ]
)

prior = tfd.JointDistributionSequential(
    [lens_prior, lens_light_prior, source_light_prior]
)

In [ ]:
# i = 79
# # prior = make_default_prior()
# kernel = np.load('/global/homes/l/linusu/gigalens/src/gigalens/assets/psf.npy').astype(np.float32)
# sim_config = SimulatorConfig(delta_pix=0.065, num_pix=80, supersample=2, kernel=kernel)
# phys_model = PhysicalModel([epl.EPL(50), shear.Shear()], [sersic.SersicEllipse(use_lstsq=False)], [sersic.SersicEllipse(use_lstsq=False), sersic.SersicEllipse(use_lstsq=False)])
# lens_sim = LensSimulator(phys_model, sim_config, bs=1)

# systems_dir = os.path.join(home, "GIGALens-Code", "SystemSaves")
# f = np.load(os.path.join(systems_dir, "100SystemsStandard80px.npz"))
# keys = f.files
# observed_imgs = jnp.array([f[key] for key in keys])
# observed_img = observed_imgs[i]

# prob_model = ForwardProbModel(prior, observed_img, background_rms=0.2, exp_time=100)
# model_seq = ModellingSequence(phys_model, prob_model, sim_config)

# normal_results_loc = os.path.join(home, f"GIGALens-Code/pipeline_results/100standard80px")
# results_dir = os.path.join(home, normal_results_loc, f"{i}")

# # results = {}
# # results["MAP"] = MAPResults.load(results_dir, model_seq)
# # results["SVI"] = SVIResults.load(results_dir, model_seq)
# # results["HMC"] = HMCResults.load(results_dir, model_seq)

In [ ]:
results = {}
map_opt = optax.adabelief(1e-2, b1=0.95, b2=0.99, nesterov=True)
best, lp, chisq = model_seq.MAP(map_opt, n_samples=500, num_steps=500)


In [ ]:
svi_opt = optax.adabelief(1e-4, b1=0.95, b2=0.99)
qz, loss_hist = model_seq.SVI(best, svi_opt, num_steps=500, n_vi=250)


In [ ]:
hmc_samples = model_seq.HMC(qz)
hmc_samples = hmc_samples.transpose((1, 2, 0, 3))
hmc_samples = hmc_samples.reshape(hmc_samples.shape[0]*hmc_samples.shape[1], *hmc_samples.shape[2:])

In [ ]:
hmc_samples.shape

In [ ]:
# cfg = PipelineConfig(steps=["MAP"], map_kwargs=dict(num_steps=350, n_samples=500))
#     # svi_kwargs=dict(num_steps=1500, n_vi=500))

# results = run_pipeline(model_seq, cfg)

In [ ]:
# results["MAP"].save(os.path.join(home, "GIGALens-Code/alternate_inference/test_system"))
# results["SVI"].save(os.path.join(home, "GIGALens-Code/alternate_inference/test_system"))
# results["HMC"].save(os.path.join(home, "GIGALens-Code/alternate_inference/test_system"))

In [ ]:
from tensorflow_probability.substrates.jax import (
    distributions as tfd,
    bijectors as tfb,
    experimental as tfe,
)
from jax import pmap, jit
def HMC_multi(
            self,
            q_z,
            init_eps=0.3,
            init_l=3,
            n_hmc=50,
            num_burnin_steps=250,
            num_results=750,
            max_leapfrog_steps=30,
            seed=0,
    ):
        dev_cnt = len(jax.devices())
        local_dev_cnt = len(jax.local_devices())
        # seeds are per process (node)
        seeds = jax.random.split(jax.random.fold_in(jax.random.PRNGKey(seed), jax.process_index()), local_dev_cnt)
        n_hmc = (n_hmc // dev_cnt) * dev_cnt
        lens_sim = LensSimulator(
            self.phys_model,
            self.sim_config,
            bs=n_hmc // dev_cnt,
        )
        momentum_distribution = tfd.MultivariateNormalFullCovariance(
            loc=jnp.zeros_like(q_z.mean()),
            covariance_matrix=jnp.linalg.inv(q_z.covariance()),
        )

        @jit
        def log_prob(z):
            return self.prob_model.log_prob(lens_sim, z)[0]

        @pmap
        def run_chain(seed):
            start = q_z.sample(n_hmc // dev_cnt, seed=seed)
            num_adaptation_steps = int(num_burnin_steps * 0.8)
            mc_kernel = tfe.mcmc.PreconditionedHamiltonianMonteCarlo(
                target_log_prob_fn=log_prob,
                momentum_distribution=momentum_distribution,
                step_size=init_eps,
                num_leapfrog_steps=init_l,
                store_parameters_in_results=True,
            )

            mc_kernel = tfe.mcmc.GradientBasedTrajectoryLengthAdaptation(
                mc_kernel,
                num_adaptation_steps=num_adaptation_steps,
                max_leapfrog_steps=max_leapfrog_steps,
            )
            mc_kernel = tfp.mcmc.DualAveragingStepSizeAdaptation(
                inner_kernel=mc_kernel, num_adaptation_steps=num_adaptation_steps
            )

            def trace_L(states, previous_kernel_results):
                # return mc_kernel.inner_kernel.num_leapfrog_steps_getter_fn(previous_kernel_results)# previous_kernel_results.num_leapfrog_steps
                return previous_kernel_results.inner_results.inner_results.proposed_results.num_leapfrog_steps

            return tfp.mcmc.sample_chain(
                num_results=num_results,
                num_burnin_steps=num_burnin_steps,
                current_state=start,
                trace_fn=trace_L,#lambda _, pkr: None,
                seed=seed,
                kernel=mc_kernel,
                # return_final_kernel_results=True,
            )

        start = time.time()
        samples = run_chain(seeds)
        end = time.time()
        # aggregate over all devices

        # print(f"LEAPFROG STEPS: {samples.num_leapfrog_steps_getter_fn(samples)}")

        # process_mesh = jax.make_mesh((local_dev_cnt,), ('local_device',))
        # sharding = jax.sharding.NamedSharding(process_mesh, P('local_device', None, None, None)) 
        # print(f'{samples.all_states.shape=}')
        # process_samples = jax.make_array_from_process_local_data(sharding, samples.all_states)
        # print(f'{process_samples.shape=}')
        # all_samples is (num_processes, num_devices_per_process, num_steps, n_hmc_per_device, 22)
        all_samples = jax.experimental.multihost_utils.process_allgather(samples.all_states)
        total_leapfrog_steps = jax.experimental.multihost_utils.process_allgather(samples.trace)
        # print(all_samples.reshape(all_samples.shape[0] * all_samples.shape[1], *all_samples.shape[2:]).shape)
        # reshape to (num_devices, num_steps, n_hmc_per_device, , 22), then swap num_steps, n_hmc_per_device
        device_partitioned_samples = jnp.swapaxes(all_samples.reshape(all_samples.shape[0] * all_samples.shape[1], *all_samples.shape[2:]), 1, 2)
        # chain_partitioned_samples is (num_chains, num_steps, 22)
        # chain_partitioned_samples = device_partitioned_samples.reshape(device_partitioned_samples.shape[0] * device_partitioned_samples.shape[1], *device_partitioned_samples.shape[2:])
        return device_partitioned_samples, total_leapfrog_steps

smp, num_leapfrog = HMC_multi(model_seq, results["SVI"].qz)

In [ ]:
# print("HMC: Total number of leapfrog steps (grad evals) per chain (varies by device):", np.sum(np.squeeze(num_leapfrog), axis=1))

In [ ]:
#! Running without SVI, using standard covaraince and MAP centroid
# init_scales = jnp.array([0.02656728, 0.01574657, 0.02841325, 0.03676566, 0.07651892,
#        0.01669485, 0.01431121, 0.01024949, 0.01149139, 0.00159058,
#        0.00066476, 0.00082975, 0.1410743 , 0.02216172, 0.00300464,
#        0.13725954, 0.01794981, 0.00236129, 0.00247942, 0.1063476 ,
#        0.11321894, 0.04579748]) / 4 #! Guess based on other system
# default_start = jnp.diag(init_scales) 

# default_start = jnp.diag(jnp.ones((best.shape[-1],))) * 1e-3
# no_SVI_qz = tfd.MultivariateNormalTriL(loc=jnp.squeeze(best), scale_tril=default_start)

# qz = no_SVI_qz
# results['SVI'].qz = no_SVI_qz

In [ ]:
def log_prob(z):
    return prob_model.log_prob(lens_sim, z)[0]

# start = jnp.squeeze(results['SVI'].qz.mean())
inv_mass_mat = qz.covariance()

# inv_mass_mat_true = jnp.cov(results['HMC'].HMC_samples_z.reshape(-1, 22).T)

transform = lambda state, info: state.position


In [ ]:
from mclmc_alt import isokinetic_mclachlan_smart, mclachlan_coefficients, mclmc_find_L_and_step_size_smart, MCLMCAdaptationState

n_grad_per_integration_step_mclachlan = len(mclachlan_coefficients)//2 #* Happens on every odd step of the integrator

integrator = isokinetic_mclachlan_smart

 # build the kernel
kernel = lambda inverse_mass_matrix : blackjax.mcmc.mclmc.build_kernel(
    logdensity_fn=log_prob,
    integrator=integrator,
    inverse_mass_matrix=inverse_mass_matrix,
)

dim =best.shape[-1]



In [ ]:
import mclmc_alt
# import mclmc_parallel
# importlib.reload(mclmc_parallel)
importlib.reload(mclmc_alt)
# from mclmc_parallel import init_multi, build_kernel_multi, mclmc_multi
from mclmc_alt import init_multi, mclmc_multi
from mclmc_alt import isokinetic_mclachlan_smart, mclmc_find_L_and_step_size_smart, MCLMCAdaptationState
import time

rng_key = jax.random.key(0)
init_key, tune_key, run_key = jax.random.split(rng_key, 3)

# start_loc = jnp.squeeze(results['SVI'].qz.sample(1, seed=init_key))

# initial_state = blackjax.mcmc.mclmc.init(
#     position=start_loc, logdensity_fn=log_prob, rng_key=init_key
# )

n_chains = 64
state_multi = init_multi(qz.sample((n_chains,), seed=init_key), init_key, log_prob)

# blackjax_state_after_tuning, blackjax_mclmc_sampler_params = burnin(tune_key, initial_state, steps=10000)
starting_adapt_state = blackjax.adaptation.mclmc_adaptation.MCLMCAdaptationState(
    jnp.sqrt(dim), jnp.sqrt(dim) * 0.25, inverse_mass_matrix=inv_mass_mat
)

starttime = time.perf_counter()
# find values for L and step_size
(
    blackjax_state_after_tuning,
    blackjax_mclmc_sampler_params,
    _
) = mclmc_find_L_and_step_size_smart(
    mclmc_kernel=kernel,
    num_steps=4000,
    state=state_multi,
    rng_key=tune_key,
    frac_tune1=0.2, #* initial step size tuning
    frac_tune2=0.6, #* Used for mass matrix adaptation
    frac_tune3=0.2, #! Tuning L. ~10 effective samples are needed for this to be accurate
    params=starting_adapt_state,
    desired_energy_var=5e-3,
    multi_chain=True,
    num_chains=n_chains,
    mass_matrix_adapt=True,
    continuous_adaptation=True,
)

total_time = time.perf_counter()-starttime
print("Burnin Time:", total_time)

In [ ]:
L = blackjax_mclmc_sampler_params.L
step_size = blackjax_mclmc_sampler_params.step_size
print(f"ADAPTED. L: {L}, step_size: {step_size}, L/step: {L/step_size}")

In [ ]:

sampling_alg = mclmc_multi(
    log_prob,
    L=L,
    step_size=step_size,
    num_chains=n_chains,
    inverse_mass_matrix=blackjax_mclmc_sampler_params.inverse_mass_matrix,
    integrator=integrator,
)

starttime = time.perf_counter()
_, multi_chain_samples = blackjax.util.run_inference_algorithm(
    rng_key=run_key,
    initial_state=blackjax_state_after_tuning,
    inference_algorithm=sampling_alg,
    num_steps=4000,
    transform=transform,
    progress_bar=True,
)

multi_chain_samples = jnp.transpose(multi_chain_samples, axes=(1, 0, 2))

total_time = time.perf_counter()-starttime
print(f"Sampling took {total_time} s")


In [ ]:
# n_grad_evals_mclmc = num_steps
n_grad_evals_mclmc = multi_chain_samples.shape[0]*multi_chain_samples.shape[1] * n_grad_per_integration_step_mclachlan
# ESS_mclmc = blackjax.diagnostics.effective_sample_size(samples_mclmc[np.newaxis, :,:], chain_axis=0, sample_axis=1)
starttime = time.perf_counter()
ESS_mclmc = blackjax.diagnostics.effective_sample_size(multi_chain_samples, chain_axis=0, sample_axis=1)
print(f"ESS Calculation took: {time.perf_counter()-starttime}")

print(ESS_mclmc)
print(np.mean(ESS_mclmc)/n_grad_evals_mclmc, " | ", jnp.min(ESS_mclmc)/n_grad_evals_mclmc)

In [ ]:
rhat = blackjax.diagnostics.potential_scale_reduction(multi_chain_samples, chain_axis=0, sample_axis=1)
print(rhat)
print(jnp.max(rhat))

In [ ]:
treedef = jax.tree.structure(prior.sample(1, seed=rng_key))
jax.tree.unflatten(treedef, rhat)

In [ ]:
hmc_rhat = blackjax.diagnostics.potential_scale_reduction(hmc_samples, chain_axis=0, sample_axis=1)
print(hmc_rhat)
print(jnp.max(hmc_rhat))

In [ ]:

sample_length = list(range(20, multi_chain_samples.shape[1], 200)) + [multi_chain_samples.shape[1]]
rhats = -np.ones((len(sample_length), multi_chain_samples.shape[-1]))
for i, l in enumerate(sample_length):
    rhats[i] = blackjax.diagnostics.potential_scale_reduction(multi_chain_samples[:, :l, :], chain_axis=0, sample_axis=1)
plt.axhline(1e-2, linestyle='--', label='Convergence')
plt.plot(sample_length, rhats-1)
plt.yscale('log')
plt.ylabel("Rhat-1")
plt.xlabel("num_results")
plt.legend()
plt.show()

In [ ]:
def bridge_sampler(rng_key, samples, log_posterior_fn, n_iter=50):
    """
    Estimates the log-marginal likelihood using the Meng-Wong bridge sampler.
    
    Args:
        rng_key: JAX random key.
        samples: MCMC samples of shape (n_samples, n_dim).
        log_posterior_fn: Function mapping (theta) -> log_posterior(theta).
        n_iter: Number of iterations for the Meng-Wong algorithm.
    """
    n_samples, n_dim = samples.shape
    
    # 1. Fit a Gaussian proposal to the MCMC samples
    mu = jnp.mean(samples, axis=0)
    cov = jnp.cov(samples, rowvar=False)
    proposal = tfd.MultivariateNormalFullCovariance(loc=mu, covariance_matrix=cov)
    
    # 2. Draw 'fake' samples from the proposal
    prop_key, _ = jax.random.split(rng_key)
    gen_samples = proposal.sample(n_samples, seed=prop_key)
    
    # 3. Evaluate log-densities (vmap for speed)
    # l1: log-posterior of MCMC samples
    # l2: log-posterior of Gen samples
    l1 = jax.vmap(log_posterior_fn)(samples)
    l2 = jax.vmap(log_posterior_fn)(gen_samples)
    # max_lp = jnp.max(l1)
    # l1 -= max_lp
    # l2 -= max_lp

    # print(l1)
    # print(l2)
    
    # q1: log-proposal of MCMC samples
    # q2: log-proposal of Gen samples
    q1 = proposal.log_prob(samples)
    q2 = proposal.log_prob(gen_samples)
    # print(q1)
    # print(q2)
    
    # 4. Iterative Meng-Wong Algorithm
    # Initial guess for the marginal likelihood (r = Z_posterior / Z_proposal)
    # Since Z_proposal = 1 (normalized), r is the evidence Z.
    log_r_guess = jnp.log(1/n_samples) + jax.nn.logsumexp(l2-q2)
    # print(log_r_guess)
    log_r = log_r_guess

    
    h = []
    s1 = 0.5 # Proportion of samples from posterior (assuming n1 == n2)
    s2 = 0.5 # Proportion of samples from proposal
    
    for _ in range(n_iter):
        # We want to compute the update: 
        # r_new = (sum(q2 / (s1*q2 + s2*r*g2))) / (sum(g1 / (s1*q1 + s2*r*g1)))
        
        # Let's define the terms for the numerator and denominator in log-space
        # Numerator term: log( q(theta_gen) / (s1*q(theta_gen) + s2*r*g(theta_gen)) )
        l_num = l2 - jnp.logaddexp(jnp.log(s1) + l2, jnp.log(s2) + log_r + q2)
        
        # Denominator term: log( g(theta_mcmc) / (s1*q(theta_mcmc) + s2*r*g(theta_mcmc)) )
        l_den = q1 - jnp.logaddexp(jnp.log(s1) + l1, jnp.log(s2) + log_r + q1)
        
        # Update log_r
        log_r_new = jax.nn.logsumexp(l_num) - jax.nn.logsumexp(l_den)
        
        # For stability, we can track the delta or use a small dampening factor if needed
        log_r = log_r_new
        h.append(log_r)

    return log_r, h

smp = hmc_samples.reshape(-1, dim)
idxes = np.random.choice(smp.shape[0], size=(1000,), replace=False)
r, hist = bridge_sampler(jax.random.key(0), smp[idxes], log_prob,n_iter=50)

In [ ]:
print(r)
plt.plot(hist)
plt.show()

log(r) for one-source model: -170.65616 (200 smp) (or -169.93636 for 1000 samples)

log(r) for two-source model: -176.59798 (200 smp) (or -175.53647 for 1000 samples) WARNING. MCLMC DID NOT CONVERGE DUE TO MULTIMODALITY. 
- HMC samples (which converged even less) got log(r)= -189.8, which is way worse, so that's a good sign

If we ignore that MCLMC didn't converge, one-source model is favored by a factor of ~e^6 = ~400. (Pretty good)


In [ ]:
# starttime = time.perf_counter()
# fake_states = np.concatenate(1000*[multi_chain_samples], axis=1)
# ess = tfp.mcmc.effective_sample_size(fake_states.transpose((1, 0, 2)), cross_chain_dims=1)
# print(f"ESS Calculation took: {time.perf_counter()-starttime}")

In [ ]:
# starttime = time.perf_counter()
# ESS_mclmc = blackjax.diagnostics.effective_sample_size(fake_states, chain_axis=0, sample_axis=1)
# print(f"ESS Calculation took: {time.perf_counter()-starttime}")

In [ ]:


def batch_means_ess(samples):
    """
    Computes a fast ESS estimate using the Batch Means method.
    Compatible with JAX jit/vmap.
    
    Args:
        samples: jnp.array of shape (num_draws,) or (num_draws, num_dims)
    Returns:
        ess: The estimated effective sample size.
    """
    # Ensure we are working with a 1D array for simplicity; 
    # use jax.vmap(batch_means_ess) for multi-chain/multi-dim
    n = samples.shape[0]
    dim = samples.shape[-1]
    
    # Determine batch size: k = sqrt(n) batches, each of size b = n/k
    # We use floor/int for JAX compatibility
    num_batches = jnp.astype(jnp.floor(jnp.sqrt(n)), 'int32')
    batch_size = n // num_batches
    
    # Truncate samples to fit perfectly into batches
    total_relevant_samples = num_batches * batch_size
    trimmed_samples = samples[:total_relevant_samples]
    
    # 1. Reshape and compute means of each batch
    batched_data = trimmed_samples.reshape((num_batches, batch_size, dim))
    batch_means = jnp.mean(batched_data, axis=1)
    
    # 2. Compute the variance of the batch means
    # We multiply by batch_size to scale it to the individual sample level
    # overall_mean = jnp.mean(trimmed_samples)
    batch_var = batch_size * jnp.var(batch_means, axis=0)
    
    # 3. Compute total process variance (unbiased)
    total_var = jnp.var(trimmed_samples, ddof=1, axis=0)
    
    # 4. ESS formula: n * (total_var / batch_var)
    # We use jnp.where to prevent division by zero
    ess = jnp.where(batch_var > 0, n * (total_var / batch_var), 1.0)
    
    return ess


def spectral_ess(samples):
    """
    ESS estimate using a Tukey-Hanning window on the spectral density.
    Much more robust than Batch Means for low-ESS chains.
    """
    n = samples.shape[0]
    # 1. Center the data
    centered = samples - jnp.mean(samples)
    
    # 2. Compute FFT-based autocovariance
    # We pad to power of 2 for maximum JAX speed
    next_pow2 = 2**jnp.ceil(jnp.log2(2*n - 1)).astype(int)
    fft_val = jnp.fft.rfft(centered, n=next_pow2)
    acf = jnp.fft.irfft(fft_val * jnp.conj(fft_val))[:n]
    acf = acf / n # Normalized by n
    
    # 3. Apply a Tukey-Hanning Window
    # The 'M' (window size) controls the bias-variance tradeoff.
    # M = sqrt(n) is a robust default.
    M = jnp.sqrt(n).astype(int)
    lags = jnp.arange(n)
    
    # Tukey-Hanning weights: 0.5 * (1 + cos(pi * lag / M)) for lag < M
    window = jnp.where(lags < M, 0.5 * (1.0 + jnp.cos(jnp.pi * lags / M)), 0.0)
    
    # 4. Estimate Integrated Autocorrelation Time (tau)
    # tau = 1 + 2 * sum(windowed_autocorrelations)
    rho = acf / acf[0]
    tau = 1.0 + 2.0 * jnp.sum(window[1:] * rho[1:])
    
    # ESS = n / tau
    return n / jnp.maximum(tau, 1.0)

single_chain = multi_chain_samples[0,:]

multi_dim_spectral_ess = jax.vmap(spectral_ess, in_axes=-1, out_axes=-1)

ess_bch = batch_means_ess(single_chain)
ess_spc = multi_dim_spectral_ess(single_chain)
ess_jax = blackjax.diagnostics.effective_sample_size(single_chain[jnp.newaxis, ...], chain_axis=0, sample_axis=1)
ess_tfp = tfp.mcmc.effective_sample_size(single_chain)
ess_jax

In [ ]:
plt.plot(ess_jax, marker='o', linestyle='', color='black', label='Blackjax ESS')
plt.plot(ess_tfp, marker='^', linestyle='', label='TFP ESS')
plt.plot(ess_bch, marker='+', linestyle='', label='Estimated ESS by batch means')
plt.plot(ess_spc, marker='*', linestyle='', label='Estimated ESS by spectral density')
plt.ylabel("ESS")
plt.title(f"Parameter ESS on a Length-{single_chain.shape[0]} Chain")
plt.ylim(0)
plt.legend()
plt.show()

# plt.plot(ess_jax, marker='o', linestyle='', color='black', label='Blackjax ESS')
plt.title(f"Error in ESS Estimation on a Length-{single_chain.shape[0]} Chain")
plt.hist((ess_spc-ess_jax)/ess_jax)
plt.xlabel('Fractional Error in Batch Means Estimated ESS')
plt.ylabel('# of Parameters')
plt.show()

In [ ]:
chain_len = results['HMC'].HMC_samples_z[0, 1].shape[0]
hmc_1chain_ESS = blackjax.diagnostics.effective_sample_size(results['HMC'].HMC_samples_z[0, 0][np.newaxis, :,:], chain_axis=0, sample_axis=1)
print(np.mean(hmc_1chain_ESS)/(chain_len*10), " | ", jnp.min(hmc_1chain_ESS)/(chain_len*10)) # 10 is guess at leapfrog steps per sample taken

In [ ]:
hmc_1chain_ESS

In [ ]:
#* Adjusted MCLMC. Doesn't seem to do as well in ESS per grad because of losses to num_integration_steps
init_key, tune_key, run_key = jax.random.split(rng_key, 3)

initial_state_adj = blackjax.mcmc.adjusted_mclmc.init(
    position=start, logdensity_fn=log_prob
)

 # build the kernel
kernel_adj = lambda rng_key, state, avg_num_integration_steps, step_size, inverse_mass_matrix : blackjax.mcmc.adjusted_mclmc.build_kernel(
    logdensity_fn=log_prob,
    integrator=blackjax.mcmc.integrators.isokinetic_mclachlan,
    inverse_mass_matrix=inverse_mass_matrix,
)(
    rng_key=rng_key,
    state=state,
    num_integration_steps=avg_num_integration_steps,
    step_size=step_size,
)

dim =22
starting_adapt_state_adj = blackjax.adaptation.adjusted_mclmc_adaptation.MCLMCAdaptationState(
        jnp.sqrt(dim), jnp.sqrt(dim) * 0.25, inverse_mass_matrix=jnp.diag(inv_mass_mat)
)

# find values for L and step_size
(
    blackjax_state_after_tuning_adj,
    blackjax_mclmc_sampler_params_adj,
    _
) = blackjax.adjusted_mclmc_find_L_and_step_size(
    mclmc_kernel=kernel_adj,
    num_steps=1000,
    state=initial_state_adj,
    rng_key=tune_key,
    target=0.9,
    diagonal_preconditioning=True,
    params=starting_adapt_state_adj
)

#* Run tuned algorithm
L = blackjax_mclmc_sampler_params_adj.L
step_size = blackjax_mclmc_sampler_params_adj.step_size
num_integration_steps = L /step_size
sampling_alg_adj = blackjax.adjusted_mclmc(
    log_prob,
    step_size=step_size,
    L_proposal_factor=L,
    inverse_mass_matrix=blackjax_mclmc_sampler_params_adj.inverse_mass_matrix,
    num_integration_steps= L/step_size
)

num_steps_adj = 1000

_, samples_mclmc_adj = blackjax.util.run_inference_algorithm(
    rng_key=run_key,
    initial_state=blackjax_state_after_tuning_adj,
    inference_algorithm=sampling_alg_adj,   
    num_steps=num_steps_adj,
    transform=transform,
    progress_bar=True,
)

In [ ]:
num_integration_steps = L /step_size
n_grad_evals_mclmc_adj = num_steps_adj * num_integration_steps
ESS_mclmc_adj = blackjax.diagnostics.effective_sample_size(samples_mclmc_adj[np.newaxis, :,:], chain_axis=0, sample_axis=1)
print(ESS_mclmc_adj)
print(np.mean(ESS_mclmc_adj)/n_grad_evals_mclmc_adj)

In [ ]:
#* Running Adjusted Dynamic MCLMC with adaptation
from blackjax.mcmc.adjusted_mclmc_dynamic import rescale
rng_key = jax.random.key(0)

init_key, tune_key, run_key = jax.random.split(rng_key, 3)

integration_steps_fn = lambda avg_num_integration_steps: lambda _: jnp.ceil(avg_num_integration_steps)

kernel = lambda rng_key, state, avg_num_integration_steps, step_size, inverse_mass_matrix: blackjax.mcmc.adjusted_mclmc_dynamic.build_kernel(
    integration_steps_fn=integration_steps_fn(avg_num_integration_steps),
    inverse_mass_matrix=inverse_mass_matrix,
)(
    rng_key=rng_key,
    state=state,
    step_size=step_size,
    logdensity_fn=log_prob,
    L_proposal_factor=jnp.inf,
)
initial_state = blackjax.mcmc.adjusted_mclmc_dynamic.init(
    position=start, logdensity_fn=log_prob, random_generator_arg=init_key
)
dim=22
starting_adapt_state = blackjax.adaptation.adjusted_mclmc_adaptation.MCLMCAdaptationState(
            jnp.sqrt(dim), jnp.sqrt(dim) * 0.25, inverse_mass_matrix=jnp.diag(inv_mass_mat),
    )

# find values for L and step_size
(
    blackjax_state_after_tuning,
    blackjax_mclmc_sampler_params,
    _
) = blackjax.adjusted_mclmc_find_L_and_step_size(
    mclmc_kernel=kernel,
    num_steps=1000,
    state=initial_state,
    rng_key=tune_key,
    target=0.9,
    diagonal_preconditioning=True,
    params=starting_adapt_state,
    # desired_energy_var=desired_energy_variance
)

step_size = blackjax_mclmc_sampler_params.step_size
L = blackjax_mclmc_sampler_params.L


sampling_alg = blackjax.adjusted_mclmc_dynamic(
    logdensity_fn=log_prob,
    step_size=step_size,
    integration_steps_fn=lambda key: jnp.ceil(
        jax.random.uniform(key) * rescale(L / step_size)
    ),
    inverse_mass_matrix=blackjax_mclmc_sampler_params.inverse_mass_matrix,
    L_proposal_factor=jnp.inf,
)

num_steps_adjust = 1000
_, samples_dynamic_adapt = blackjax.util.run_inference_algorithm(
    rng_key=run_key,
    initial_state=blackjax_state_after_tuning,
    inference_algorithm=sampling_alg,
    num_steps=num_steps_adjust,
    transform=transform,
    progress_bar=True,
)

In [ ]:
n_grad_evals_adjust = num_steps_adjust * jnp.ceil(L/step_size)
ESS_adjust = blackjax.diagnostics.effective_sample_size(samples_dynamic_adapt[np.newaxis, :,:], chain_axis=0, sample_axis=1)
print(np.mean(ESS_adjust)/n_grad_evals_adjust)


In [ ]:
from tensorflow_probability.substrates.jax import (
    distributions as tfd,
    bijectors as tfb,
    experimental as tfe,
)
def NUTS_multi(
        self,
        q_z,
        init_eps=0.3,
        # init_l=3,
        n_hmc=50,
        num_burnin_steps=250,
        num_results=750,
        # max_leapfrog_steps=30,
        seed=0,
    ):
    
    dev_cnt = len(jax.devices())
    local_dev_cnt = len(jax.local_devices())
    # seeds are per process (node)
    seeds = jax.random.split(jax.random.fold_in(jax.random.PRNGKey(seed), jax.process_index()), local_dev_cnt)
    n_hmc = (n_hmc // dev_cnt) * dev_cnt
    lens_sim = LensSimulator( # sim.
        self.phys_model,
        self.sim_config,
        bs=n_hmc // dev_cnt,
    )
    momentum_distribution = tfd.MultivariateNormalFullCovariance(
        loc=jnp.zeros_like(q_z.mean()),
        covariance_matrix=jnp.linalg.inv(q_z.covariance()),
    )

    @jax.jit
    def log_prob(z):
        return self.prob_model.log_prob(lens_sim, z)[0]

    @jax.pmap
    def run_chain(seed):
        start = q_z.sample(n_hmc // dev_cnt, seed=seed)
        num_adaptation_steps = int(num_burnin_steps * 0.8)
        mc_kernel = tfe.mcmc.PreconditionedNoUTurnSampler(
            target_log_prob_fn=log_prob,
            step_size=init_eps,
            momentum_distribution=momentum_distribution,
        )

        # mc_kernel = tfe.mcmc.GradientBasedTrajectoryLengthAdaptation(
        #     mc_kernel,
        #     num_adaptation_steps=num_adaptation_steps,
        #     max_leapfrog_steps=max_leapfrog_steps,
        # )
        # mc_kernel = tfp.mcmc.DualAveragingStepSizeAdaptation(
        #     inner_kernel=mc_kernel, num_adaptation_steps=num_adaptation_steps
        # )

        return tfp.mcmc.sample_chain(
            num_results=num_results,
            # num_burnin_steps=num_burnin_steps,
            current_state=start,
            trace_fn=lambda _, pkr: None,
            seed=seed,
            kernel=mc_kernel,
        )

    start = time.time()
    samples = run_chain(seeds)
    end = time.time()
    # aggregate over all devices

    # process_mesh = jax.make_mesh((local_dev_cnt,), ('local_device',))
    # sharding = jax.sharding.NamedSharding(process_mesh, P('local_device', None, None, None)) 
    # print(f'{samples.all_states.shape=}')
    # process_samples = jax.make_array_from_process_local_data(sharding, samples.all_states)
    # print(f'{process_samples.shape=}')
    # all_samples is (num_processes, num_devices_per_process, num_steps, n_hmc_per_device, 22)
    all_samples = jax.experimental.multihost_utils.process_allgather(samples.all_states)
    # print(all_samples.reshape(all_samples.shape[0] * all_samples.shape[1], *all_samples.shape[2:]).shape)
    # reshape to (num_devices, num_steps, n_hmc_per_device, , 22), then swap num_steps, n_hmc_per_device
    device_partitioned_samples = jnp.swapaxes(all_samples.reshape(all_samples.shape[0] * all_samples.shape[1], *all_samples.shape[2:]), 1, 2)
    # chain_partitioned_samples is (num_chains, num_steps, 22)
    # chain_partitioned_samples = device_partitioned_samples.reshape(device_partitioned_samples.shape[0] * device_partitioned_samples.shape[1], *device_partitioned_samples.shape[2:])
    return device_partitioned_samples

In [ ]:
nuts_results = NUTS_multi(model_seq, results["SVI"].qz, init_eps=0.05, n_hmc=8,num_burnin_steps=100,num_results=100,)

In [ ]:
#* Run basic HMC (no burnin)
num_integration_steps = 10
hmc = blackjax.hmc(
    log_prob, step_size=0.05, inverse_mass_matrix=jnp.diag(inv_mass_mat_true), num_integration_steps=num_integration_steps
)
hmc_init_state = hmc.init(start)

num_steps_hmc = 1000

_, samples_hmc = blackjax.util.run_inference_algorithm(
    rng_key=run_key,
    initial_state=hmc_init_state,
    inference_algorithm=hmc,
    num_steps=num_steps_hmc,
    transform=transform,
    progress_bar=True,
)

n_grad_evals_hmc = num_steps_hmc * num_integration_steps
ESS_hmc = blackjax.diagnostics.effective_sample_size(samples_hmc[np.newaxis, :,:], chain_axis=0, sample_axis=1)
print(np.mean(ESS_hmc)/n_grad_evals_hmc)

In [ ]:
#* Run basic NUTS

nuts = blackjax.nuts(
    log_prob, step_size=0.05, inverse_mass_matrix=jnp.diag(inv_mass_mat)
)
state = nuts.init(start)
# step = jax.jit(nuts.step)

# num_grad_evals_nuts = 0
# for i in range(100):
#     state, info = step(rng_key, state)
#     num_grad_evals_nuts += info.num_integration_steps
    

_, samples_nuts = blackjax.util.run_inference_algorithm(
    rng_key=run_key,
    initial_state=state,
    inference_algorithm=nuts,
    num_steps=1000,
    transform=transform,
    progress_bar=True,
)

ESS_nuts = blackjax.diagnostics.effective_sample_size(samples_mclmc[np.newaxis, :,:], chain_axis=0, sample_axis=1)


In [ ]:
print(np.mean(ESS_nuts/(10*6684)))

In [ ]:
#### print("L:", params.L, "Step Size:", params.step_size)
print(np.abs(blackjax_mclmc_sampler_params.inverse_mass_matrix - jnp.diag(inv_mass_mat))/jnp.diag(inv_mass_mat))

In [ ]:
# save_loc = os.path.join(home, f"GIGALens-Code/pipeline_results/100standard80px_mclmc")
# save_dir = os.path.join(home, save_loc, f"{i}")
# mclmc_results_100sys = HMCResults.load(save_dir, model_seq)

In [ ]:
samples = multi_chain_samples[:, :,:].reshape(-1, dim) #samples_mclmc 
MCMC_x = prob_model.bij.forward(list(samples.T))

adapt_cov = blackjax_mclmc_sampler_params.inverse_mass_matrix
adapt_qz = tfd.MultivariateNormalFullCovariance(loc=jnp.mean(samples, axis=0), covariance_matrix=adapt_cov)
adapt_qz_samples = adapt_qz.sample((10000,), run_key)
adapt_qz_x = prob_model.bij.forward(list(adapt_qz_samples.T))

SVI_samples = qz.sample((1000,), run_key)
SVI_x = prob_model.bij.forward(list(SVI_samples.T))

# HMC_1chain_x = prob_model.bij.forward(list(results['HMC'].HMC_samples_z[0, 11, -40000:].T))

plot_params = cornerplot_labels(MCMC_x)

hmc_x = prob_model.bij.forward(list(hmc_samples.reshape(-1, dim).T))

n_samp = hmc_samples.shape[0]*hmc_samples.shape[1]
rand_idx = np.random.choice(np.arange(n_samp), size=(10000,), replace=False)

fig = cornerplot_posterior(jax.tree.map(lambda x: x[rand_idx], hmc_x), color='black',plot_params=plot_params)

n_samp_MCMC = MCMC_x[0][0]['e1'].shape[0]
rand_idx_MCMC = np.random.choice(np.arange(n_samp_MCMC), size=(20000,), replace=True)
cornerplot_posterior(jax.tree.map(lambda x: x[rand_idx_MCMC], MCMC_x), fig=fig, color='red',plot_params=plot_params)

# cornerplot_posterior(SVI_x, fig=fig, color='blue', plot_params=plot_params)
cornerplot_posterior(adapt_qz_x, fig=fig, color='purple')
# cornerplot_posterior(mclmc_results_100sys.HMC_samples, fig=fig, color='purple', plot_params=plot_params)
# cornerplot_posterior(HMC_1chain_x, fig=fig, color='green', plot_params=plot_params)

plt.show()

In [ ]:
fig, ax = plt.gcf(), plt.gca()
plot_image(fig, ax, observed_img)
plt.show()

In [ ]:
ess_nuts_tf = blackjax.diagnostics.effective_sample_size(nuts_results.reshape((8, 100, 22)), chain_axis=0, sample_axis=1)
print(ess_nuts_tf)

In [ ]:
results['HMC'].HMC_samples_z.shape

In [ ]:
ess_nuts = blackjax.diagnostics.effective_sample_size(samples[np.newaxis, :,:], chain_axis=0, sample_axis=1)
print(ess_nuts)

In [ ]:
ess_hmc = blackjax.diagnostics.effective_sample_size(results['HMC'].HMC_samples_z.reshape((48, 750, 22)), chain_axis=0, sample_axis=1)
print(np.mean(ess_hmc)/(750*30))

In [ ]:
print(ess_nuts/ess_hmc)

In [ ]:
np.mean(ess_nuts/ess_hmc)